**Question** What statistical and structural signals are present in the eight input features?
**Method** Plot distributions, correlations, fault-type profiles, and the configured dependency graph using exported data and topology configuration.
**Takeaway** EDA describes what the models can observe; it does not substitute for held-out evaluation.

In [ ]:
from pathlib import Path
import sys, yaml
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
ROOT=Path.cwd(); sys.path.extend([str(ROOT),str(ROOT/'notebooks')])
from notebooks.common.plotting import style
style(); node=pd.read_csv(ROOT/'data/node_timeseries.csv'); episodes=pd.read_csv(ROOT/'data/episodes.csv')
features=['cpu_utilization','memory_utilization','p95_latency','error_rate','in_degree','out_degree','criticality','queue_depth']
fig, axes=plt.subplots(2,4,figsize=(14,6))
for ax, feature in zip(axes.flat, features): sns.histplot(data=node, x=feature, hue='is_root_cause', stat='density', common_norm=False, element='step', ax=ax); ax.set_title(feature)
fig.tight_layout(); plt.show()

### What this tells us
These distributions show overlap and scale for the actual generated observations. Root-cause coloring is descriptive only; no thresholds are selected here.

In [ ]:
episode_features=node.groupby('episode_id')[features].mean()
plt.figure(figsize=(9,7)); sns.heatmap(episode_features.corr(), cmap='vlag', center=0, square=True); plt.title('Episode-level feature correlation'); plt.show()
profile=node.merge(episodes[['episode_id','fault_type']],on='episode_id').groupby('fault_type')[features].mean()
display(profile)

### What this tells us
The heatmap and fault profiles expose redundant signals and fault-specific signatures that may be useful for interpretation. They are computed at episode level so a long time series cannot dominate the correlation merely by having more rows.

In [ ]:
topology=yaml.safe_load((ROOT/'configs/topology.yaml').read_text())
fig, ax=plt.subplots(figsize=(9,4)); positions={n['id']:(i,0) for i,n in enumerate(topology['nodes'])}
for n in topology['nodes']:
    for dst in n.get('dependencies',[]): ax.annotate('',xy=positions[dst],xytext=positions[n['id']],arrowprops={'arrowstyle':'->','lw':1.5})
for node_id,(x,y) in positions.items(): ax.scatter(x,y,s=900); ax.text(x,y,node_id,ha='center',va='center',color='white')
ax.set_axis_off(); ax.set_title('Configured directed dependency graph'); plt.show()

### What this tells us
The graph is read from the topology YAML rather than embedded in notebook code. Direction matters because the propagation label follows downstream reachability in the repository’s graph preprocessing path.

## Feature meanings
CPU and memory are normalized utilization; p95 latency is capped and normalized; error rate is normalized; in-degree and out-degree describe dependency position; criticality is the graph structural score; queue depth is continuous backpressure telemetry. The model sees no categorical health-status proxy.